# Phase 5: Comprehensive Evaluation

In [ ]:
import sys
import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

if "google.colab" in sys.modules:
    if "/content" not in sys.path:
        sys.path.insert(0, "/content")
else:
    sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

from project_paths import *

In [ ]:
def load_npz(path):
    data = np.load(path)
    return data['X'], data['Y']

x_train, y_train = load_npz(PROCESSED_DATA_DIR / "train" / "sequences.npz")
x_val, y_val = load_npz(PROCESSED_DATA_DIR / "validation" / "sequences.npz")
x_test, y_test = load_npz(PROCESSED_DATA_DIR / "test" / "sequences.npz")

print(f"Train shape: {x_train.shape} -> {y_train.shape}")
print(f"Val shape: {x_val.shape} -> {y_val.shape}")
print(f"Test shape: {x_test.shape} -> {y_test.shape}")

history_steps = x_test.shape[1]
horizon_steps = y_test.shape[1]
nodes = x_test.shape[2]
features = x_test.shape[3]

print(f"\nExtracted Config:")
print(f"History: {history_steps}")
print(f"Horizon: {horizon_steps}")
print(f"Nodes: {nodes}")
print(f"Features: {features}")

In [ ]:
from src.data.dataset import get_dataloaders

batch_size = 64
train_loader, val_loader, test_loader = get_dataloaders(
    x_train, y_train,
    x_val, y_val,
    x_test, y_test,
    batch_size=batch_size
)

batch_x, batch_y = next(iter(test_loader))
print(f"Batch X: {batch_x.shape}")
print(f"Batch Y: {batch_y.shape}")

## Evaluate All Models & Plot

In [ ]:
from src.models.baseline import HistoricalAverage, LastValue
from src.evaluation.evaluate import evaluate_model_comprehensive
from src.evaluation.visualization import plot_actual_vs_predicted, plot_error_by_horizon

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ha_model = HistoricalAverage(horizon_steps=horizon_steps)
lv_model = LastValue(horizon_steps=horizon_steps)

print("Evaluating...")
ha_results = evaluate_model_comprehensive(ha_model, test_loader, device)
lv_results = evaluate_model_comprehensive(lv_model, test_loader, device)
print("Done.")

In [ ]:
results_df = pd.DataFrame([
    {'model': 'HA', **ha_results['overall']},
    {'model': 'LV', **lv_results['overall']},
])
results_df